### Permutation Feature Importance

特徴量の1つだけをランダムにシャッフル $\rightarrow$ オリジナルのデータと予測精度を比較

これをすべての特徴量に試して影響の大きい特徴量を抽出

In [2]:
!pip install japanize_matplotlib

zsh:1: /Users/shimizutoorushin/Desktop/大学\M-h\M-,-義/M1\M-f|(業program/M1_Class/.venv/bin/pip: bad interpreter: /Users/shimizutoorushin/Desktop/M1_Class_program/M1_Class/.venv/bin/python: no such file or directory
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-pa

#### シミュレーション

モデル

$Y = 0X_0 + 1X_1 + 2X_2 + \varepsilon$

$\begin{pmatrix}
   X_0 \\
   X_1 \\
   X_2
\end{pmatrix} \sim {\mathcal N}
\begin{pmatrix}
\begin{pmatrix}
   0 \\
   0 \\
   0
\end{pmatrix}, 
\begin{pmatrix}
   1 & 0 & 0 \\
   0 & 1 & 0 \\
   0 & 0 & 1
\end{pmatrix}
\end{pmatrix}
$

$\varepsilon \sim {\mathcal N} (0, 0.01)$

In [3]:
# 3次元シミュレーションデータを線形重回帰で予測
# 当然説明変数は3つ

import sys
import warnings
from dataclasses import dataclass
from typing import Any  # 型ヒント用
from __future__ import annotations  # 型ヒント用
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # matplotlibの日本語表示対応
from mli.visualize import get_visualization_setting  # 自作モジュール

np.random.seed(42)
pd.options.display.float_format = "{:.2f}".format
sns.set(**get_visualization_setting())
warnings.simplefilter("ignore")  # warningsを非表示に

from sklearn.model_selection import train_test_split


def generate_simulation_data(N, beta, mu, Sigma):
    """線形のシミュレーションデータを生成し、訓練データとテストデータに分割する
    
    Args: 
        N: インスタンスの数
        beta: 各特徴量の傾き
        mu: 各特徴量は多変量正規分布から生成される。その平均。
        Sigma: 各特徴量は多変量正規分布から生成される。その分散共分散行列。
    """

    # 多変量正規分布からデータを生成
    
    X = np.random.multivariate_normal(mu, Sigma, N)

    # ノイズは平均0標準偏差0.1(分散は0.01)で決め打ち
    
    epsilon = np.random.normal(0, 0.1, N)

    # 特徴量とノイズの線形和で目的変数を作成
    
    y = X @ beta + epsilon

    return train_test_split(X, y, test_size=0.2, random_state=42)


# シミュレーションデータの設定

N = 1000
J = 3
mu = np.zeros(J)
Sigma = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
beta = np.array([0, 1, 2])

# シミュレーションデータの生成

X_train, X_test, y_train, y_test = generate_simulation_data(N, beta, mu, Sigma)

def plot_scatter(X, y, var_names):
    """目的変数と特徴量の散布図を作成"""

    # 特徴量の数だけ散布図を作成
    
    J = X.shape[1]
    fig, axes = plt.subplots(nrows=1, ncols=J, figsize=(4 * J, 4))

    for d, ax in enumerate(axes):
        sns.scatterplot(x=X[:, d], y=y, alpha=0.3, ax=ax)
        ax.set(
            xlabel=var_names[d], 
            ylabel="Y", 
            xlim=(X.min() * 1.1, X.max() * 1.1)
        )

    fig.show()

# 可視化

var_names = ["X({0:d})".format(j) for j in range(J)]
plot_scatter(X_train, y_train, var_names)

# X0はほぼ無相関
# X2が最も相関が強くなるのは当然

ModuleNotFoundError: No module named 'japanize_matplotlib'

In [ ]:
# 線形重回帰の偏回帰係数を可視化

from sklearn.linear_model import LinearRegression

def plot_bar(variables, values, title=None, xlabel=None, ylabel=None):
    """回帰係数の大きさを確認する棒グラフを作成"""
    
    fig, ax = plt.subplots()
    ax.barh(variables, values)
    ax.set(xlabel=xlabel, ylabel=ylabel, xlim=(0, None))
    fig.suptitle(title)
    
    fig.show()


# 線形回帰モデルの学習

lm = LinearRegression().fit(X_train, y_train)

# 回帰係数の可視化

plot_bar(var_names, lm.coef_, "線形回帰の回帰係数の大きさ", "回帰係数")

# X2の偏回帰係数が最大なのも当然

In [ ]:
# PFIのお手製の実装

from sklearn.metrics import mean_squared_error


@dataclass
class PermutationFeatureImportance:
    """Permutation Feature Importance (PFI)
     
    Args:
        estimator: 全特徴量を用いた学習済みモデル
        X: 特徴量
        y: 目的変数
        var_names: 特徴量の名前
    """
    
    estimator: Any
    X: np.ndarray
    y: np.ndarray
    var_names: list[str]
        
    def __post_init__(self) -> None:
        # シャッフルなしの場合の予測精度
        # mean_squared_error()はsquared=TrueならMSE、squared=FalseならRMSE
        self.baseline = mean_squared_error(
            self.y, self.estimator.predict(self.X), squared=False
        )

    def _permutation_metrics(self, idx_to_permute: int) -> float:
        """ある特徴量の値をシャッフルしたときの予測精度を求める

        Args:
            idx_to_permute: シャッフルする特徴量のインデックス
        """

        # シャッフルする際に、元の特徴量が上書きされないよう用にコピーしておく
        X_permuted = self.X.copy()

        # 特徴量の値をシャッフルして予測
        X_permuted[:, idx_to_permute] = np.random.permutation(
            X_permuted[:, idx_to_permute]
        )
        y_pred = self.estimator.predict(X_permuted)

        return mean_squared_error(self.y, y_pred, squared=False)

    def permutation_feature_importance(self, n_shuffle: int = 10) -> None:
        """PFIを求める

        Args:
            n_shuffle: シャッフルの回数。多いほど値が安定する。デフォルトは10回
        """

        J = self.X.shape[1]  # 特徴量の数

        # J個の特徴量に対してPFIを求めたい
        # R回シャッフルを繰り返して平均をとることで値を安定させている
        metrics_permuted = [
            np.mean(
                [self._permutation_metrics(j) for r in range(n_shuffle)]
            )
            for j in range(J)
        ]

        # データフレームとしてまとめる
        # シャッフルでどのくらい予測精度が落ちるかは、
        # 差(difference)と比率(ratio)の2種類を用意する
        df_feature_importance = pd.DataFrame(
            data={
                "var_name": self.var_names,
                "baseline": self.baseline,
                "permutation": metrics_permuted,
                "difference": metrics_permuted - self.baseline,
                "ratio": metrics_permuted / self.baseline,
            }
        )

        self.feature_importance = df_feature_importance.sort_values(
            "permutation", ascending=False
        )

    def plot(self, importance_type: str = "difference") -> None:
        """PFIを可視化

        Args:
            importance_type: PFIを差(difference)と比率(ratio)のどちらで計算するか
        """

        fig, ax = plt.subplots()
        ax.barh(
            self.feature_importance["var_name"],
            self.feature_importance[importance_type],
            label=f"baseline: {self.baseline:.2f}",
        )
        ax.set(xlabel=importance_type, ylabel=None)
        ax.invert_yaxis() # 重要度が高い順に並び替える
        ax.legend(loc="lower right")
        fig.suptitle(f"Permutationによる特徴量の重要度({importance_type})")
        
        fig.show()

In [ ]:
# ランダムフォレストで予測モデルを作成
# 予測精度を確認

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train, y_train)

# 予測精度を確認

print("R2: {0:.2f}".format(r2_score(y_test, rf.predict(X_test))))

# 予測精度はかなり高め

In [ ]:
# PFIを計算して可視化

# PFIのインスタンスの作成

pfi = PermutationFeatureImportance(rf, X_test, y_test, var_names)

# PFIを計算

pfi.permutation_feature_importance()

# PFIを可視化

pfi.plot(importance_type="difference")

# 一応まともに動作している様子

### Leave One Covariate Out Feature Importance (LOCOFI)

訓練データとテストデータの両方である特徴量のデータを黒塗りして予測

全特徴量での予測精度と比較して誤差が大きければ重要な特徴量と認識

### Group Permutation Feature Importance (GPFI)

強く相関する特徴量はまとめてシャッフル

特徴量を群と捉えた方が解釈しやすいケースもある

カテゴリカル変数の場合もGPFIの方が良いらしい

In [ ]:
# GPFIのお手製メソッド

class GroupedPermutationFeatureImportance(PermutationFeatureImportance):
    """Grouped Permutation Feature Importance (GPFI)"""

    def _permutation_metrics(
        self,
        var_names_to_permute: list[str]
    ) -> float:
        """ある特徴量群の値をシャッフルしたときの予測精度を求める

        Args:
            var_names_to_permute: シャッフルする特徴量群の名前
        """

        # シャッフルする際に、元の特徴量が上書きされないよう用にコピーしておく
        X_permuted = self.X.copy()

        # 特徴量名をインデックスに変換
        idx_to_permute = [
            self.var_names.index(v) for v in var_names_to_permute
        ]

        # 特徴量群をまとめてシャッフルして予測
        X_permuted[:, idx_to_permute] = np.random.permutation(
            X_permuted[:, idx_to_permute]
        )
        y_pred = self.estimator.predict(X_permuted)

        return mean_squared_error(self.y, y_pred, squared=False)

    def permutation_feature_importance(
        self,
        var_groups: list[list[str]] | None = None,
        n_shuffle: int = 10
    ) -> None:
        """GPFIを求める

        Args:
            var_groups:
                グループ化された特徴量名のリスト。例：[['X0', 'X1'], ['X2']]
                Noneを指定すれば通常のPFIが計算される
            n_shuffle:
                シャッフルの回数。多いほど値が安定する。デフォルトは10回
        """

        # グループが指定されなかった場合は1つの特徴量を1グループとする。PFIと同じ。
        if var_groups is None:
            var_groups = [[j] for j in self.var_names]

        # グループごとに重要度を計算
        # R回シャッフルを繰り返して値を安定させている
        metrics_permuted = [
            np.mean(
                [self._permutation_metrics(j) for r in range(n_shuffle)]
            )
            for j in var_groups
        ]

        # データフレームとしてまとめる
        # シャッフルでどのくらい予測精度が落ちるかは、差と比率の2種類を用意する
        df_feature_importance = pd.DataFrame(
            data={
                "var_name": ["+".join(j) for j in var_groups],
                "baseline": self.baseline,
                "permutation": metrics_permuted,
                "difference": metrics_permuted - self.baseline,
                "ratio": metrics_permuted / self.baseline,
            }
        )

        self.feature_importance = df_feature_importance.sort_values(
            "permutation", ascending=False
        )

In [ ]:
# シミュレーションデータに適用

# 特徴量X2と全く同じ特徴量を追加 (X3)

X_train2 = np.concatenate([X_train, X_train[:, [2]]], axis=1)

# 特徴量X2と全く同じ特徴量を追加した新しいデータからRandom Forestの予測モデルを構築

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train2, y_train)

# テストデータにも同様に特徴量X2とまったく同じ値をとる特徴量X3を作る。

X_test2 = np.concatenate([X_test, X_test[:, [2]]], axis=1)

gpfi = GroupedPermutationFeatureImportance(rf, X_test2, y_test, ["X0", "X1", "X2", "X3"])

# var_groupsを指定しなければ通常のPFIが計算される

gpfi.permutation_feature_importance()

# 可視化

gpfi.plot()

# X2をコピーしたX3も重要度は高め

In [ ]:
# X2とX3はまとめてシャッフル。X0とX1は個別にシャッフル

gpfi.permutation_feature_importance(var_groups=[["X0"], ["X1"], ["X2", "X3"]])

# GPFIを可視化

gpfi.plot()

# X2+X3が非常に高くなるのは狙い通り

#### 擬似相関

特徴量のペアに相関があると目的変数との相関を見誤ることもある

特徴量重要度を因果関係に使わない方が良い理由の1つ

モデル

$Y = X_0 + \varepsilon$

$\begin{pmatrix}
   X_0 \\
   X_1 \\
   X_2
\end{pmatrix} \sim {\mathcal N}
\begin{pmatrix}
\begin{pmatrix}
   0 \\
   0 \\
   0
\end{pmatrix}, 
\begin{pmatrix}
   1 & 0.95 & 0 \\
   0.95 & 1 & 0 \\
   0 & 0 & 1
\end{pmatrix}
\end{pmatrix}
$

$\varepsilon \sim {\mathcal N} (0, 0.01)$

$X_1$も$X_2$も$Y$には影響しない

ただ$X_0$と$X_1$には相関があるという想定

In [ ]:
# シミュレーションデータの設定

N = 1000
J = 3
mu = np.zeros(J)
Sigma = np.array([[1, 0.95, 0], [0.95, 1, 0], [0, 0, 1]])
beta = np.array([1, 0, 0])

# シミュレーションデータの生成

X_train, X_test, y_train, y_test = generate_simulation_data(N, beta, mu, Sigma)

# 全特徴量を使ってRandom Forestの予測モデルを構築

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train, y_train)

# PFIを計算

pfi = PermutationFeatureImportance(rf, X_test, y_test, var_names)
pfi.permutation_feature_importance()

# PFIを可視化

pfi.plot(importance_type="difference")

# X0だけが重要であるとまともに認識している

In [ ]:
# X0は使わずRandom Forestの予測モデルを構築

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train[:, [1, 2]], y_train)

# X0は使わないでPFIを計算

pfi = PermutationFeatureImportance(rf, X_test[:, [1, 2]], y_test, ["X1", "X2"])
pfi.permutation_feature_importance()

# PFIを可視化

pfi.plot(importance_type="difference")

# X1に重要度がついてしまっている

#### 重要度の検討に訓練/テストデータのどちらを使うべきか

過学習を起こしている場合は訓練/テストデータで差が出るはず

過学習でない場合はどちらも用いても大差はなさそう

#### PFIの利点

機械学習のモデルによらず同じ方法で重要度を計算可能

アプローチが直感的に理解しやすい

計算時間も割と短め

#### PFIの注意点

強く相関する特徴量で重要度を食い合うケースがある $\rightarrow$ 特徴量をまとめてシャッフル

PFIだけで因果分析をするのは危険

### Partial Dependence (PD)

注目する特徴量をいろいろ変更 (摂動)

他の特徴量はそのまま

各インスタンスの予測値を平均して可視化

"可視化"もアルゴリズムの一部なのでPartial Dependence Plot (PDP)と呼ばれることも

シミュレーションモデル1

$Y = X + \varepsilon$

$X \sim {\rm U}(0, 1)$

$\varepsilon \sim {\mathcal N} (0, 0,01)$

In [ ]:
import sys
import warnings
from dataclasses import dataclass
from typing import Any  # 型ヒント用
from __future__ import annotations  # 型ヒント用
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # matplotlibの日本語表示対応

# 自作モジュール

from mli.visualize import get_visualization_setting

np.random.seed(42)
pd.options.display.float_format = "{:.2f}".format
sns.set(**get_visualization_setting())
warnings.simplefilter("ignore")  # warningsを非表示に

from sklearn.model_selection import train_test_split


def generate_simulation_data1():
    """シミュレーション1のデータを生成"""

    N = 1000  # インスタンス数
    beta = np.array([1])  # 回帰係数

    X = np.random.uniform(0, 1, [N, 1])  # 一様分布から特徴量を生成
    epsilon = np.random.normal(0, 0.1, N)  # 正規分布からノイズを生成
    y = X @ beta + epsilon  # 線形和で目的変数を作成

    return train_test_split(X, y, test_size=0.2, random_state=42)


# シミュレーションデータの生成

X_train, X_test, y_train, y_test = generate_simulation_data1()

def plot_scatter(x, y, xlabel="X", ylabel="Y", title=None):
    """散布図を作成"""
    
    fig, ax = plt.subplots()
    sns.scatterplot(x=x, y=y, alpha=0.3, ax=ax)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    fig.suptitle(title)
    fig.show()


#　特徴量Xと目的変数Yの散布図を作成 

plot_scatter(X_train[:, 0], y_train, title="XとYの散布図")

In [ ]:
# 線形回帰で予測モデルを作成

from sklearn.linear_model import LinearRegression
from mli.metrics import regression_metrics


# 線形回帰モデルの学習

lm = LinearRegression().fit(X_train, y_train)

# 予測精度を確認

regression_metrics(lm, X_test, y_test)

In [ ]:
from mli.utility import get_coef

# 切片と特徴量Xの回帰係数を確認

df_coef = get_coef(lm, ['X'])
df_coef.T

シミュレーションモデル2

$Y = 10 \sin (X_0) + X_1 + \varepsilon$

$X_0 \sim {\rm U} (-2\pi, 2\pi)$

$X_1 \sim {\rm U} (-2\pi, 2\pi)$

$\varepsilon \sim {\mathcal N} (0, 0.01)$

In [ ]:
def generate_simulation_data2():
    """シミュレーション2のデータを生成"""

    N = 1000  # インスタンス数

    # 一様分布から特徴量を生成
    X = np.random.uniform(-np.pi * 2, np.pi * 2, [N, 2])
    epsilon = np.random.normal(0, 0.1, N)  # 正規分布からノイズを生成

    # 特徴量X0はsin関数で変換する
    y = 10 * np.sin(X[:, 0]) + X[:, 1] + epsilon

    return train_test_split(X, y, test_size=0.2, random_state=42)


# シミュレーションデータの生成

X_train, X_test, y_train, y_test = generate_simulation_data2()

def plot_scatters(X, y, var_names, title=None):
    """目的変数と特徴量の散布図を作成"""
    
    # 特徴量の数だけ散布図を作成
    J = X.shape[1]
    fig, axes = plt.subplots(nrows=1, ncols=J, figsize=(4 * J, 4))

    for j, ax in enumerate(axes):
        sns.scatterplot(x=X[:, j], y=y, alpha=0.3, ax=ax)
        ax.set(
            xlabel=var_names[j], 
            ylabel="Y", 
            xlim=(X.min() * 1.1, X.max() * 1.1)
        )
    fig.suptitle(title)
    
    fig.show()


# 特徴量ごとに目的変数との散布図を作成

plot_scatters(X_train, y_train, ["X0", "X1"], title="特徴量と目的変数の散布図")

# X_0とYの関係は当然非線形

In [ ]:
# 線形重回帰モデルの学習

lm = LinearRegression().fit(X_train, y_train)

# 予測精度の確認

regression_metrics(lm, X_test, y_test)

# 決定係数が低いのは想定通り

In [ ]:
# 切片と特徴量X0, X1の回帰係数を確認

df_coef = get_coef(lm, ['X0', 'X1'])
df_coef.T

In [ ]:
# ランダムフォレストによる予測モデルの構築

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train, y_train)

# 予測精度の確認

regression_metrics(rf, X_test, y_test)

# ランダムフォレストだと決定係数がまとも

In [ ]:
# インスタンス0を取り出す

i = 0
Xi = X_test[[i]]

# 特徴量を出力

print(Xi)

In [ ]:
# インスタンス0に対する予測値

print("(X0, X1)=(-0.37, -5.67)のときの予測値：{0:.2f}".format(rf.predict(Xi)[0]))

In [ ]:
# 特徴量の置き換えをお手製メソッドで試す

def counterfactual_prediction(estimator, X, idx_to_replace, value_to_replace):
    """ある特徴量の値を置き換えたときの予測値を求める

    Args:
        estimator: 学習済みモデル
        X: 特徴量
        idx_to_replace: 値を置き換える特徴量のインデックス
        value_to_replace: 置き換える値
    """
    
    # 特徴量の値を置き換える際に、元の特徴量が上書きされないよう用にコピーしておく
    X_replaced = X.copy()

    # 特徴量の値を置き換えて予測
    X_replaced[:, idx_to_replace] = value_to_replace
    y_pred = estimator.predict(X_replaced)

    return y_pred


# X0の値を-4に置き換えた場合の予測値

cp = counterfactual_prediction(rf, Xi, 0, -4)[0]
print("(X0, X1)=(-4, -5.67)のときの予測値：{0:.2f}".format(cp))

# X0の値をグッと下げたら予測値増加

In [ ]:
# X0の値を-3に置き換えた場合の予測値を出力

cp = counterfactual_prediction(rf, Xi, 0, -3)[0]
print("(X0, X1)=(-3, -5.67)のときの予測値：{0:.2f}".format(cp))

In [ ]:
# X0の取りうる範囲を50個に分割

X0_range = np.linspace(-np.pi * 2, np.pi * 2, num=50)

# 取りうる範囲でX0の値を動かして予測値を生成

cps = np.concatenate(
    [counterfactual_prediction(rf, Xi, 0, x) for x in X0_range]
)
cps

In [ ]:
# よくわからないので可視化

def plot_line(x, y, xlabel="X", ylabel="Y", title=None):
    """特徴量の値を変化させた場合の予測値の推移を可視化"""
    
    fig, ax = plt.subplots()
    ax.plot(x, y)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    fig.suptitle(title)
    
    fig.show()

# 可視化

plot_line(X0_range, cps, "X0", "モデルの予測値", "インスタンス{0:d}に関する特徴量X0と予測値の関係".format(i))

# サインカーブっぽい

In [ ]:
# インスタンス10を取り出す

i = 10
Xi = X_test[[i]]

# 特徴量を出力

Xi

In [ ]:
# インスタンス10に対する予測値

print("(X0, X1)=(4.52, -0.52)のときの予測値：{0:.2f}".format(rf.predict(Xi)[0]))

In [ ]:
# インスタンス10についてもX0の値を動かして予測値を生成

cps = np.concatenate([counterfactual_prediction(rf, Xi, 0, x) for x in X0_range])

# 可視化

plot_line(X0_range, cps, "X0", "モデルの予測値", "インスタンス{0:d}に関する特徴量X0と予測値の関係".format(i))

# インスタンス0と比較すると予測値が小さくなっている

In [ ]:
# すべてのインスタンスに対して予測値を出し、インスタンスごとの結果を平均する

avg_cps = np.array([counterfactual_prediction(rf, X_test, 0, x).mean() for x in X0_range])

# 可視化

plot_line(X0_range, avg_cps, "X0", "モデルの予測値の平均", "特徴量X0と予測値の平均的な関係")

# ほぼサインカーブ
# これが「平均化して可視化する」PDの仕組み

In [ ]:
# PDのお手製クラス

@dataclass
class PartialDependence:
    """Partial Dependence (PD)

    Args:
        estimator: 学習済みモデル
        X: 特徴量
        var_names: 特徴量の名前
    """
    
    estimator: Any
    X: np.ndarray
    var_names: list[str]
    
    def _counterfactual_prediction(
        self,
        idx_to_replace: int,
        value_to_replace: float
    ) -> np.ndarray:
        """ある特徴量の値を置き換えたときの予測値を求める

        Args:
            idx_to_replace: 値を置き換える特徴量のインデックス
            value_to_replace: 置き換える値
        """

        # 特徴量の値を置き換える際、元データが上書きされないようコピー
        X_replaced = self.X.copy()

        # 特徴量の値を置き換えて予測
        X_replaced[:, idx_to_replace] = value_to_replace
        y_pred = self.estimator.predict(X_replaced)

        return y_pred

    def partial_dependence(
        self,
        var_name: str,
        n_grid: int = 50
    ) -> None:
        """PDを求める

        Args:
            var_name: 
                PDを計算したい特徴量の名前
            n_grid: 
                グリッドを何分割するか
                細かすぎると値が荒れるが、粗すぎるとうまく関係を捉えられない
                デフォルトは50
        """
        
        # 可視化の際に用いるのでターゲットの変数名を保存
        self.target_var_name = var_name  
        # 変数名に対応するインデックスをもってくる
        var_index = self.var_names.index(var_name)

        # ターゲットの変数を、取りうる値の最大値から最小値まで動かせるようにする
        value_range = np.linspace(
            self.X[:, var_index].min(), 
            self.X[:, var_index].max(), 
            num=n_grid
        )

        # インスタンスごとのモデルの予測値を平均
        average_prediction = np.array([
            self._counterfactual_prediction(var_index, x).mean()
            for x in value_range
        ])

        # データフレームとしてまとめる
        self.df_partial_dependence = pd.DataFrame(
            data={var_name: value_range, "avg_pred": average_prediction}
        )

    def plot(self, ylim: list[float] | None = None) -> None:
        """PDを可視化

        Args:
            ylim: 
                Y軸の範囲
                特に指定しなければavg_predictionの範囲となる
                異なる特徴量のPDを比較したいときなどに指定する
        """

        fig, ax = plt.subplots()
        ax.plot(
            self.df_partial_dependence[self.target_var_name],
            self.df_partial_dependence["avg_pred"],
        )
        ax.set(
            xlabel=self.target_var_name,
            ylabel="Average Prediction",
            ylim=ylim
        )
        fig.suptitle(f"Partial Dependence Plot ({self.target_var_name})")
        
        fig.show()
        
# PDのインスタンスを作成
# pandasとかぶるので変数名はpdp(partial dependence plot)とした

pdp = PartialDependence(rf, X_test, ["X0", "X1"])

# X1に対するPDを計算

pdp.partial_dependence("X1", n_grid=50)

# PDを可視化

pdp.plot()

# X1が1増えたら予測値も1程度増加

#### Partial Dependenceの数式的理解

特徴量: $X_0, X_1$

学習済みモデル: ${\hat f} (X_0, X_1)$

特徴量$X_0 = x_0$のときの平均的な予測値

$\widehat{\rm PD}_0 (x_0) = \frac 1 N \sum^N_{i=1} {\hat f} (x_0, x_{i, 1})$

一般化して${\bf X} = (X_1, \ldots, X_J)$を入力とした学習済みモデル${\hat f}({\bf X})$

注目している特徴量$X_j$ それ以外の特徴量${\bf X}_{\backslash j} = (X_1, \ldots, X_{j-1}, X_{j+1}, \ldots, X_J)$

インスタンス$i$の特徴量$j$の実測値$x_{i, j}$，特徴量$j$以外の実測値${\bf x}_{i, \backslash j} = (x_{i, 1}, \ldots, x_{i, j-1}, x_{i, j+1}, \ldots, x_{i, J})$

$X_j = x_j$の場合の平均的な予測値

$\widehat{\rm PD}_j (x_j) = \frac 1 N \sum^N_{i=1} {\hat f} (x_j, {\bf x}_{i, \backslash j})$

Partial Dependence Function

${\rm PD}_j (x_j) = {\mathbb E} [{\hat f} (x_j, {\bf X}_{\backslash j})] = \int {\hat f} (x_j, {\bf X}_{\backslash j}) p ({\bf x}_{\backslash j}) d{\bf x}_{\backslash j}$

ex. ${\hat f} (X_0, X_1) = {\hat \beta}_0 X_0 + {\hat \beta}_1 X_1$の場合

$X_0 = x_0$に関するPartial Dependence Function

${\rm PD}_0 (x_0) = {\mathbb E} [{\hat f} (x_0, X_1)] = {\mathbb E} [{\hat \beta}_0 x_0 + {\hat \beta}_1 X_1] = {\hat \beta}_0 x_0 + {\hat \beta}_1 {\mathbb E} [X_1]$

シミュレーションモデル3

$Y = X_1 + \varepsilon$

$\begin{pmatrix}
   X_0 \\
   X_1 
\end{pmatrix} \sim {\mathcal N} (
\begin{pmatrix}
   0 \\
   0 
\end{pmatrix}, 
\begin{pmatrix}
   1 & 0.95 \\
   0.95 & 1
\end{pmatrix}
)$

$\varepsilon \sim {\mathcal N} (0, 0.01)$

In [ ]:
def generate_simulation_data3():
    """シミュレーション3のデータを生成"""

    N = 1000  # インスタンス数
    beta = np.array([0, 1])  # 回帰係数

    # 多変量正規分布から強く相関するデータを生成
    mu = np.array([0, 0])
    Sigma = np.array([[1, 0.95], [0.95, 1]])
    X = np.random.multivariate_normal(mu, Sigma, N)
    epsilon = np.random.normal(0, 0.1, N)  # 正規分布からノイズを生成
    y = X @ beta + epsilon  # 線形和で目的変数を作成

    return train_test_split(X, y, test_size=0.2, random_state=42)

# シミュレーションデータの生成

X_train, X_test, y_train, y_test = generate_simulation_data3()

# X0と散布図を作成

plot_scatter(X_train[:, 0], y_train, xlabel="X0", title="X0とYの散布図")

# 本来X0とYには因果関係はない
# 相関の強いX1がYと因果関係があるだけ

In [ ]:
# ランダムフォレストで予測

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train, y_train)

# 予測精度の確認

regression_metrics(rf, X_test, y_test)

In [ ]:
# PDのインスタンスを作成

pdp = PartialDependence(rf, X_test, ["X0", "X1"])

# X0に対するPDを計算

pdp.partial_dependence("X0", n_grid=50)

# PDを可視化

pdp.plot(ylim=(y_train.min(), y_train.max()))

# X0を変えても予測値に全く影響なし

In [ ]:
# X1に対するPDを計算、可視化

pdp.partial_dependence("X1", n_grid=50)
pdp.plot(ylim=(y_train.min(), y_train.max()))

# X1の方は値を変化させたら予測値に影響が出る

#### PDを因果関係に使えるか?

予測モデルが特徴量と目的変数の関係をうまく捉えていれば使える

そうでない場合は間違った結論になる危険性あり

In [ ]:
# X0だけでYを予測するモデルだとどうなるか

# モデルの学習

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train[:, [0]], y_train)

# PDのインスタンスを作成

pdp = PartialDependence(rf, X_test[:, [0]], ["X0"])

# X0に対するPDを計算

pdp.partial_dependence("X0", n_grid=50)

# PDを可視化

pdp.plot(ylim=(y_train.min(), y_train.max()))

# X0の変化が予測値に影響を与えている
# これだけを見るとX0とYの間に因果関係があるように錯覚してしまう

In [ ]:
# 予測精度

regression_metrics(rf, X_test[:, [0]], y_test)

# 決定係数がそれほど悪くないのが厄介